# TrainLM T1.4: single-host TorchTPU distributed probe

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Dhiraj309/TrainLM/blob/docs/01-t1-v5e1-probe/notebooks/TorchTPU_T1_v5e8_Distributed_Probe.ipynb)

Open this notebook in Colab to run or copy its code cells directly, including on mobile. This distributed probe requires a suitable v5e-8 runtime.

Use a v5e-8 TPU host for the optional collective. This is a bounded feasibility check, not distributed training or T4 certification. Run the single-device T1 notebook first. The run stays disabled until you choose to test it. Record the actual VM/accelerator identity and package versions with the result.


In [ ]:
from pathlib import Path
import json
import os
import runpy
import subprocess
import sys
import tempfile

PROBE_BRANCH = "docs/01-t1-v5e1-probe"
REPOSITORY_URL = "https://github.com/Dhiraj309/TrainLM.git"
# Each setup run gets a fresh snapshot; existing checkouts are preserved.
session_dir = Path(tempfile.mkdtemp(prefix="trainlm-t1-"))
repo_root = session_dir / "TrainLM"
subprocess.run(
    ["git", "clone", "--depth", "1", "--single-branch", "--branch", PROBE_BRANCH,
     REPOSITORY_URL, str(repo_root)], check=True, timeout=120,
)
SOURCE_REVISION = subprocess.check_output(
    ["git", "-C", str(repo_root), "rev-parse", "HEAD"], text=True,
).strip()
subprocess.run(["git", "-C", str(repo_root), "checkout", "--detach", SOURCE_REVISION],
               check=True, timeout=30)


def assert_source():
    revision = subprocess.check_output(
        ["git", "-C", str(repo_root), "rev-parse", "HEAD"], text=True,
    ).strip()
    changes = subprocess.check_output(
        ["git", "-C", str(repo_root), "status", "--porcelain"], text=True,
    ).strip()
    if revision != SOURCE_REVISION or changes:
        raise RuntimeError("Probe source changed. Rerun setup to start a new evidence session.")


assert_source()
RUN_PROCESS = runpy.run_path(str(repo_root / "src/trainlm/runtime/torchtpu_probe_process.py"))["run_probe_process"]
EVIDENCE = runpy.run_path(str(repo_root / "src/trainlm/runtime/torchtpu_evidence.py"))
T1_RESULTS = dict.fromkeys(("package_process", "eager_process", "compile_process",
                          "lifecycle_report", "distributed_process"))
NOTEBOOK_HOST = ("Colab" if "COLAB_RELEASE_TAG" in os.environ or "google.colab" in sys.modules
                 else "Kaggle" if "KAGGLE_KERNEL_RUN_TYPE" in os.environ or "KAGGLE_URL_BASE" in os.environ
                 else "other")


def run_script(script, arguments=(), *, timeout=300, shutdown_grace=5):
    assert_source()
    result = RUN_PROCESS([sys.executable, str(repo_root / script), *arguments],
                         cwd=repo_root, timeout=timeout, shutdown_grace=shutdown_grace)
    assert_source()
    return result


print(f"notebook_host={NOTEBOOK_HOST}")
print(f"source_revision={SOURCE_REVISION}")
print("Fresh source snapshot ready; previous notebook results cleared.")

## Read-only package preflight

Record the installed profile for this distributed session too. This checks source and TorchTPU metadata plus `pip check`; it does not install packages or initialize a TPU. A mismatch remains visible in the downloaded report.

In [ ]:
T1_RESULTS["package_process"] = None
T1_RESULTS["package_process"] = run_script(
    "examples/torchtpu_t1_install_preflight.py", timeout=60,
)
print(json.dumps(EVIDENCE["profile_evidence"](T1_RESULTS["package_process"]), indent=2))

## Read-only launch-environment inspection

An `incomplete` result is normal before TorchTPU prepares its launch environment. This cell does not import torch or initialize a TPU. Coordinator addresses are never printed.


In [ ]:
inspection = run_script("examples/torchtpu_t1_distributed_probe.py", ["--inspect"], timeout=15)
print(inspection["stdout"], end="")
if inspection["status"] != "completed":
    raise RuntimeError("Launch-environment inspection failed.")

## Optional v5e-8 collective

Set `RUN_DISTRIBUTED = True` only on an isolated single-host v5e-8 runtime after the single-device smoke passes. TrainLM calls TorchTPU's launch helper in the parent process, starts eight workers, verifies rank/world consistency and one sum all-reduce, and tears the worker group down. It times out after 300 seconds. A failure is evidence to investigate; do not claim distributed support from a skipped run.


In [ ]:
RUN_DISTRIBUTED = False
T1_RESULTS["distributed_process"] = None
if RUN_DISTRIBUTED:
    # The outer deadline includes preparation and allows the inner launcher to clean up.
    T1_RESULTS["distributed_process"] = run_script(
        "examples/torchtpu_t1_distributed_probe.py",
        ["--run", "--expected-world-size", "8", "--timeout", "300"],
        timeout=360, shutdown_grace=15,
    )
    distributed = EVIDENCE["distributed_evidence"](T1_RESULTS["distributed_process"])
    print(json.dumps(distributed, indent=2))
    if distributed["status"] != "passed":
        raise RuntimeError("Distributed check failed; download the report below before investigating.")
else:
    print("distributed=not_run; enable only on an isolated single-host v5e-8 runtime")

## Download a shareable result

Run this cell even after a failed check. Colab offers a JSON download; other notebook hosts display a file link. The report includes the exact source revision, package comparison, normalized hardware observations and outcomes. Skipped checks stay `not_run`, and CPU results stay labeled as non-TPU evidence. Raw logs, environment variables, coordinator addresses and local artifact paths are excluded.

Share this JSON together with the selected runtime image and actual accelerator identity. This report requires owner review and does not certify native training, gradient-correct data parallelism, or performance. Rerun setup to clear all results and use a fresh source snapshot.

In [ ]:
assert_source()
report = EVIDENCE["build_evidence"](
    source_revision=SOURCE_REVISION, notebook_host=NOTEBOOK_HOST, **T1_RESULTS,
)
EVIDENCE_PATH = Path.cwd() / "trainlm_t1_evidence.json"
EVIDENCE_PATH.write_text(json.dumps(report, indent=2, sort_keys=True) + "\n", encoding="utf-8")
print(json.dumps({name: check["status"] for name, check in report["checks"].items()}, indent=2))
try:
    from google.colab import files
except ImportError:
    from IPython.display import FileLink, display
    display(FileLink(EVIDENCE_PATH.name))
else:
    files.download(str(EVIDENCE_PATH))